# 02 · Parquet with Hardwood

`data/celestrak_gp_catalog.parquet` holds 21 207 satellites and 52 columns. The
point of a columnar file is that you rarely need to read all of it. Hardwood
gives you three levels, cheapest first:

1. **the footer** — shape and statistics, from the last few kilobytes
2. **row readers** — stream rows through a filter, one at a time
3. **column readers** — whole columns as primitive arrays

This notebook climbs those three levels.

In [0]:
addDependency("dev.hardwood:hardwood-core:1.1.0.Beta1")

import dev.hardwood.InputFile;
import dev.hardwood.internal.predicate.StatisticsDecoder;
import dev.hardwood.reader.ColumnReader;
import dev.hardwood.reader.FilterPredicate;
import dev.hardwood.reader.ParquetFileReader;
import dev.hardwood.reader.RowReader;
import dev.hardwood.schema.ColumnProjection;

String num(double v) { return String.format(java.util.Locale.ROOT, "%,.0f", v); }
String dec(double v, int p) { return String.format(java.util.Locale.ROOT, "%." + p + "f", v); }

var catalog = ParquetFileReader.open(InputFile.of(cwd.resolve("data/celestrak_gp_catalog.parquet")));
var meta = catalog.getFileMetaData();

## 1 · The footer

Everything here comes from the file trailer. No data page is decompressed, which
is why it costs the same on a 13 MB file and a 13 GB one.

In [0]:
println("rows        %s", num(meta.numRows()));
println("row groups  %d", meta.rowGroups().size());
println("written by  %s", meta.createdBy());

## 2 · The schema

The schema lists every column. Count them — and note how many are the `satcat_*`
enrichment columns added on top of the raw element sets.

In [0]:
var schema = catalog.getFileSchema();
int satcatCols = 0;
for (var c : schema.getColumns()) if (c.name().startsWith("satcat_")) satcatCols++;
println("columns %d, of which %d are satcat_*", schema.getColumnCount(), satcatCols);

## 3 · Streaming the rows

A row reader hands you one row at a time. Count the debris objects — a row is
debris when its `satcat_object_type` is `DEB` or `R/B` — using a single `long` of
state, never a list.

In [0]:
long debris = 0;
try (RowReader rows = catalog.rowReader()) {
    while (rows.hasNext()) {
        rows.next();
        String t = rows.getString("satcat_object_type");
        if ("DEB".equals(t) || "R/B".equals(t)) debris++;
    }
}
println("debris objects: %s", num(debris));

## 4 · Column batches

A column reader returns a whole batch as one primitive array — no row object, no
boxing. Find the highest inclination in the catalog by touching a `double[]`.

In [0]:
double maxInc = Double.NEGATIVE_INFINITY;
try (ColumnReader col = catalog.columnReader("inclination")) {
    while (col.nextBatch()) {
        double[] values = col.getDoubles();
        int n = col.getValueCount();
        for (int k = 0; k < n; k++) if (values[k] > maxInc) maxInc = values[k];
    }
}
println("highest inclination: %s deg", dec(maxInc, 2));

## 5 · Statistics without touching the data

Each column chunk stores the minimum and maximum it saw. Read them straight out
of the footer — the values never leave the trailer. They arrive as raw bytes and
`StatisticsDecoder` decodes them per physical type.

In [0]:
var chunk = meta.rowGroups().get(0);
for (var c : chunk.columns()) {
    if (!c.metaData().pathInSchema().leafName().equals("inclination")) continue;
    var stats = c.metaData().statistics();
    println("inclination  %s .. %s (from the footer only)",
            dec(StatisticsDecoder.decodeDouble(stats.minValue()), 3),
            dec(StatisticsDecoder.decodeDouble(stats.maxValue()), 3));
}

## 6 · given — where the file's bytes actually live

Still no data page read: every column chunk records how it was encoded and how
many bytes it took. This is the columnar payoff in one table.

In [0]:
// ── given ─────────────────────────────────────────────────────────────
var wanted = java.util.Set.of("norad_cat_id", "mean_motion", "eccentricity",
        "inclination", "altitude_km", "period_min", "orbit_class", "norad_groups");

println("%-16s %-7s %11s %11s %7s", "column", "codec", "stored", "raw", "ratio");
for (var c : meta.rowGroups().get(0).columns()) {
    var cm = c.metaData();
    String name = cm.pathInSchema().leafName();
    if (!wanted.contains(name)) continue;
    println("%-16s %-7s %11s %11s %6sx", name, cm.codec(),
            num(cm.totalCompressedSize()), num(cm.totalUncompressedSize()),
            dec((double) cm.totalUncompressedSize() / cm.totalCompressedSize(), 1));
}

In [0]:
// ── check ─────────────────────────────────────────────────────────────
var pass = meta.numRows() == 21207 && catalog.getFileSchema().getColumnCount() == 52 && debris == 3501 && maxInc > 142.0 && maxInc < 142.1;
if (pass) {
    println("✓ 21,207 rows · 52 columns · %s debris · max inclination %s", num(debris), dec(maxInc, 2));
} else {
    println("✗ something is off — check the TODOs above");
}

**Which level to reach for:** footer for *shape*, row readers for *which
rows*, column readers for *what the values are*. Records and lists only when you
need random access.

Next: [`03-dataframes-with-dflib.ipynb`](03-dataframes-with-dflib.ipynb).